In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# 📊 Phase 1: Exploratory Data Analysis (EDA)

In this initial phase, we assess the structural characteristics, scale, and feature distributions of the **Smart MCQ Solver Challenge** dataset directly within the Kaggle environment.

---

## 📐 1. Dataset Dimensions & Integrity
We evaluate the physical shape (number of samples and features) and inspect for missing data points across both splits.

### 📝 Train Dataset Statistics
* **Total Records (Questions):** `train_df.shape[0]`
* **Total Features (Columns):** `train_df.shape[1]`
* **Missing Values:** Checked via `train_df.isnull().sum()`

### 📝 Test Dataset Statistics
* **Total Records (To Predict):** `test_df.shape[0]`
* **Total Features (Columns):** `test_df.shape[1]`
* **Missing Values:** Checked via `test_df.isnull().sum()`

---

## 🔎 2. Feature & Schema Identification
Understanding the layout of the text variables to map our data pipeline inputs.


| Column Name | Data Type | Role in Pipeline | Description |
| :--- | :--- | :--- | :--- |
| **`ID`** | Integer / Object | Identifier | Unique token tracking every specific question |
| **`prompt`** | Text (String) | Model Input | The core question context or premise |
| **`A`, `B`, `C`, `D`, `E`** | Text (String) | Model Input | The 5 alternative answer variations |
| **`answer`** | Character (String) | Target Label | The ground truth letter (Only in Train data) |

---

## 🎯 3. Target Distribution (Answer Bias Analysis)
We measure the frequency of each ground truth class (`A`, `B`, `C`, `D`, `E`) in the training set to check for systemic target imbalances. 
* *Note:* This historical distribution serves as the logical backbone for our Mode-based baseline.


In [2]:
# Here we use the pd.shape method to essentially get an idea as to how large of a dataset we are working with
# As Kaggle already has the pandas method imported we need not import the same everytime
train_path = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'
test_path = '/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv'

# Converting the Paths given by Kaggle to a Dataframe
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Gettng the shape from the dataframe
trn_shp = train_df.shape
tst_shp = test_df.shape

# Displaying the results 
print(f"Traing Data shape -> {trn_shp}")
print(f"Testing Data shape -> {tst_shp}")

Traing Data shape -> (2000, 8)
Testing Data shape -> (500, 7)


In [3]:
# We peek on the data so as to identify a sample as to how the dataset looks like
# We use the pd.head() method to do the same
HD_VAL = 5
hd_dst = train_df.head(HD_VAL)
print(hd_dst)

   id                                             prompt  \
0   1  Pick the best possible answer: What is Martin ...   
1   2        What is accelerator-based light-ion fusion?   
2   3  Determine the correct option: What is the term...   
3   4  Select the most accurate option: What is Marti...   
4   5  Identify the correct statement: What is the co...   

                                                   A  \
0  Martin Heidegger believes that humans exist wi...   
1  Accelerator-based light-ion fusion is a techni...   
2                                       Blueshifting   
3  Martin Heidegger believes that humans exist wi...   
4  Simultaneity is relative, meaning that two eve...   

                                                   B  \
0  Martin Heidegger believes that humans do not e...   
1  Accelerator-based light-ion fusion is a techni...   
2                                        Redshifting   
3  Martin Heidegger believes that humans do not e...   
4  Simultaneity is rel

In [4]:
# Getting the information about the type of data in the dataset 
# This is done by using the .info() and the .describe() method
print(train_df.info())
print("="*50)
print(train_df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB
None
                id
count  2000.000000
mean   1000.500000
std     577.494589
min       1.000000
25%     500.750000
50%    1000.500000
75%    1500.250000
max    2000.000000


In [5]:
# Now we find the value counts of the objective answer to see which one comes maximum as a bid to make the baseline submission
vci = train_df['answer'].value_counts()
print(f'The Value counts are -> \n{vci}')

The Value counts are -> 
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64


## First Dummy Submission

As a starting point, we created a simple baseline submission without using any machine learning model. The idea was to predict the most frequent answer choices observed in the training dataset for every question in the test set.

### Methodology

1. Compute the frequency of each answer option (`A`, `B`, `C`, `D`, `E`) in the training data.
2. Select the three most common answer choices.
3. Use these three choices as the prediction for every test sample.
4. Generate a submission file in the required competition format.

### Rationale

This baseline serves as a sanity check for the submission pipeline and establishes a minimum benchmark score. Any subsequent machine learning model should outperform this dummy strategy.

### Result

The baseline submission achieved a Kaggle score of **0.34871**, providing a reference point for evaluating future models.

In [6]:
# Making our submission as a dummy (Baseline)
# Creating the top 3 most occuring options (the modal values) then converting them to a list
# After we convert to a list we essentially convert to a string value thats necessary for the submission format
t_3 = train_df['answer'].value_counts().index[:3].tolist()
pred = ' '.join(t_3)
print(f"The Baseline Prediction is -> {pred}")

# And now we make a DataFrame for the dummy submission
dummy_df = pd.DataFrame({
    'ID' : test_df['id'],
    'Prediction' : pred
})

# Now we convert the DataFrame to a CSV (comma separated values) for the submission format
dummy_df.to_csv(
    'submission.csv',
    index = False
)
print("Baseline Sumbission CSV is here !!")

The Baseline Prediction is -> B C A
Baseline Sumbission CSV is here !!


## Phase 1: Classical Machine Learning Models

### Model 1: Logistic Regression with TF-IDF

Following the baseline submission, the first actual machine learning approach employed was a combination of TF-IDF vectorization and Logistic Regression. This serves as a strong classical Natural Language Processing baseline and is widely used for text classification tasks.

### Methodology

#### 1. Text Construction

The question prompt and all five answer options were concatenated into a single text field for each sample. This allowed the model to consider the complete context of the question while learning.

#### 2. TF-IDF Vectorization

The textual data was transformed into numerical feature vectors using the Term Frequency–Inverse Document Frequency (TF-IDF) technique.

Key configuration:

- Maximum vocabulary size: **5000**
- Vocabulary learned only from the training set
- Test set transformed using the same learned vocabulary

TF-IDF emphasizes informative words while reducing the influence of commonly occurring terms.

#### 3. Model Training

A Logistic Regression classifier was trained on the TF-IDF features generated from the training data.

Key configuration:

- Maximum iterations: **1000**
- Multiclass classification handled internally by the algorithm

The model learns linear decision boundaries within the high-dimensional TF-IDF feature space to distinguish between the possible answer classes.

#### 4. Prediction Generation

For every test sample, the model produced class probabilities for each answer option. The top three most probable classes were selected and formatted according to the competition submission requirements.

### Rationale

This approach represents a significant improvement over the frequency-based baseline by incorporating both linguistic information and supervised learning. Despite its simplicity, Logistic Regression combined with TF-IDF remains a highly competitive benchmark for many text classification tasks.

### Result

The TF-IDF + Logistic Regression model achieved a Kaggle score of **0.74064**, more than doubling the performance of the baseline submission and establishing a strong benchmark for subsequent ensemble and boosting-based approaches.

In [7]:
# Now for the second and 1st proper submission we use the TFIdfVectorizer and the Loistic Regression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression as lgr

# Combining all the text into one string per row
train_df['text'] = train_df['prompt'] + ' ' + train_df['A'] + ' ' + train_df['B'] + ' ' + train_df['C'] + ' ' + train_df['D'] + ' ' + train_df['E']
test_df['text'] =  test_df['prompt']  + ' ' + test_df['A']  + ' ' + test_df['B']  + ' ' + test_df['C']  + ' ' + test_df['D']  + ' ' + test_df['E']

# TFIdf now converts text to numbers suitably
tfidf = TfidfVectorizer(
    max_features = 5000 # this only keeps the 5000 most important words. Ignores rare words to reduce noise.
)
x_train = tfidf.fit_transform(train_df['text']) # learns vocab FROM train, then converts
x_test = tfidf.transform(test_df['text']) # only converts, doesn't learn again

# Now Giving the labels
y_train = train_df['answer']

# Now here we train Logistic Regression
lr_model = lgr(
    max_iter = 1000 # maximum number of times the model adjusts itself while learning. Default
)
lr_model.fit(
    x_train,
    y_train
)

# Predict probabilities for each class
probs = lr_model.predict_proba(
    x_test  # predict_proba → gives as a probability for each classes
)
clas = lr_model.classes_
print("Logistic Regression and TFIdf initialized properly")

# Get the top 3 predictions per question
t3_pr = []
for pr in probs:
    top3_idx = pr.argsort()[::-1][ :3]  # argsort() — sorts by index from lowest to highest probability
    top3_lbls = ' '.join(clas[top3_idx])
    t3_pr.append(top3_lbls)
print("The model predictions are out")
print("The model's work is successful here")

Logistic Regression and TFIdf initialized properly
The model predictions are out
The model's work is successful here


In [8]:
# Creating the submission dataframe
submission_lr = pd.DataFrame({
    'ID' : test_df['id'],
    'Prediction' : t3_pr
})

print("Created the DataFrame Successfully ...")

# Converting the above DataFrame to a CSV output file
submission_lr.to_csv(
    'submission.csv',
    index = False
)
print(f"The Submission CSV is out\nThe Features in the DataFrame are :->\n{submission_lr.head(10)}")

Created the DataFrame Successfully ...
The Submission CSV is out
The Features in the DataFrame are :->
   ID Prediction
0   1      A C E
1   2      B A C
2   3      B D C
3   4      E C D
4   5      C A D
5   6      D C A
6   7      E C B
7   8      B A D
8   9      C D B
9  10      B C D


In [9]:
# Now we try to use the CatBoost Algorithm
# Why is CatBoost better than other Boosting and Ensemble methods (This is because CatBoost handles Textual and Categorical data directly)
from catboost import CatBoostClassifier as cbcl, Pool

# Training Pool is made to take raw textual data
train_pool = Pool(
    data = train_df['text'],
    label = train_df['answer'],
    text_features = [0]
)
print("Training Data is pooled successfully !! ")

# Testual Data is Pooled the same way as in as training data 
test_pool = Pool(
    data = test_df['text'],
    text_features = [0]
)
print("Test Data is pooled successfully !!")

Training Data is pooled successfully !! 
Test Data is pooled successfully !!


## Phase 2: Gradient Boosting Models

### Model 2: Categorical Boosting (CatBoost) Classifier

After establishing a strong benchmark using Logistic Regression and TF-IDF, the next step was to explore gradient boosting techniques specifically designed for handling textual and categorical data. For this purpose, the CatBoost classifier was selected.

### Methodology

#### 1. Text Construction

As in the previous approach, the question prompt and all answer choices were combined into a single textual representation for each sample. This ensured that the model had access to the complete context of the question while making predictions.

#### 2. Native Text Processing

Unlike traditional machine learning algorithms, CatBoost is capable of processing raw textual data directly. The textual inputs were therefore supplied through CatBoost's `Pool` data structure, which allows the framework to internally perform text feature extraction and representation learning.

This removed the need for explicit vectorization techniques such as TF-IDF.

#### 3. Training Pool Creation

The training dataset was converted into a CatBoost `Pool` containing:

- The combined textual input
- The target answer labels
- Metadata identifying the text feature column

A similar pool was created for the test dataset to ensure consistent preprocessing.

#### 4. Model Training

A multiclass CatBoost classifier was trained using the following configuration:

- Iterations: **500**
- Learning Rate: **0.10**
- Depth: **6**
- Loss Function: **MultiClass**

The model was trained directly on the raw textual data and optimized using gradient boosting over decision trees.

#### 5. Probability-Based Predictions

For each test sample, CatBoost generated class probabilities corresponding to all possible answer options.

The three answer choices with the highest probabilities were selected and combined into the required competition submission format.

### Rationale

CatBoost was chosen because of its ability to natively process textual information without requiring manual feature engineering. By integrating text handling and gradient boosting into a single framework, it offers a powerful alternative to traditional TF-IDF based pipelines.

### Result

The CatBoost classifier achieved a Kaggle score of **0.74979**, outperforming the TF-IDF + Logistic Regression approach and becoming the strongest model evaluated during this phase of experimentation.

### Observations

- CatBoost successfully leveraged raw textual information without external vectorization.
- The model demonstrated improved predictive performance over classical linear methods.
- Training times were significantly higher than Logistic Regression, but the increase in leaderboard performance justified the additional computational cost.
- The experiment highlighted the effectiveness of boosting-based approaches for this text classification task.

In [10]:
# # Here we instantiate and train CatBoost 
# cb_model = cbcl(
#     iterations = 500,
#     learning_rate = 0.1,
#     depth = 6,
#     loss_function = 'MultiClass',
#     verbose = 100 
# )

# # Here we fit the model
# cb_model.fit(train_pool)
# print("CatBoost is Trained with the training data")

# # Here we predict the possiblities
# probs_cb = cb_model.predict_proba(test_pool)
# classes_cb = cb_model.classes_

# # Now we get the 3 prediction per question
# t3_pr_cb = []
# for prob in probs_cb:
#     top3_idx = prob.argsort()[::-1][:3]
#     top3_lbls = ' '.join(classes_cb[top3_idx])
#     t3_pr_cb.append(top3_lbls)

# print("The final model is ready and its predictions are out")

In [11]:
# # Creating the submission dataframe
# submission_cb = pd.DataFrame({
#     'ID' : test_df['id'],
#     'Prediction' : t3_pr_cb
# })

# # Now Converting the same dataframe to a CSV format
# submission_cb.to_csv(
#     'submission.csv',
#     index = False
# )
# print(f"The Final CSV formatted DataFrame is out\nIt Looks like this :->\n{submission_cb.head(10)}")

## Phase 3: Transformers and Pre-Trained Language Models

### Model 1: BERT-Based Zero-Shot Classification

Having explored both classical machine learning and gradient boosting approaches, the next stage of experimentation focused on Transformer-based language models. These models are pre-trained on massive text corpora and possess strong language understanding capabilities.

For this experiment, a pre-trained Transformer model was utilized through the Hugging Face Transformers library to perform zero-shot text classification.

### Methodology

#### 1. Pre-Trained Language Model

A pre-trained Natural Language Inference (NLI) model was loaded using the Hugging Face `pipeline` interface.

Model used:

- `cross-encoder/nli-MiniLM2-L6-H768`

The model was downloaded from the Hugging Face Model Hub and loaded for inference.

#### 2. Zero-Shot Classification

Unlike previous approaches, no task-specific training was performed.

Instead, the model leveraged knowledge acquired during pre-training to directly evaluate candidate answer labels for each question.

This approach is commonly referred to as **Zero-Shot Learning**, where predictions are generated without additional fine-tuning on the target dataset.

#### 3. Inference Pipeline

For each question:

- The prompt and answer choices were supplied to the model.
- Candidate labels corresponding to the answer options were evaluated.
- The model assigned confidence scores to each candidate answer.
- The top three highest-scoring labels were selected for submission.

#### 4. Submission Generation

The top-ranked answer choices were formatted according to the competition requirements and exported as a submission file.

### Rationale

Transformer models have demonstrated state-of-the-art performance across a wide variety of Natural Language Processing tasks. This experiment aimed to investigate whether a general-purpose pre-trained language model could effectively solve the task without requiring additional training.

### Result

The zero-shot Transformer approach achieved a Kaggle score of **0.41064**.

### Observations

- The model required no task-specific training.
- Inference was computationally expensive compared to classical machine learning methods.
- Despite strong general language understanding capabilities, the zero-shot approach significantly underperformed both Logistic Regression and CatBoost.
- The results suggest that task-specific training or fine-tuning is necessary to fully exploit the potential of Transformer architectures for this competition.

In [12]:
# # Next for our third model we use a pretrained model as per the submissions
# # We use BERT for this
# from transformers import pipeline

# # This is essentially a 'Download' of the model from HuggingFace
# classifier = pipeline(
#     'zero-shot-classification',
#     model = 'cross-encoder/nli-MiniLM2-L6-H768',
#     device = -1  # Runs on CPU load
# )
# print("Model Loaded successfully !!")

# # Candidate labels are essentially constant
# candidate_labels = ['A', 'B' , 'C' , 'D' , 'E']

# # Getting the top3 prediction per question
# t3_per_bert = []

# for idx, row in test_df.iterrows():
#     # Builds the context -> question + all options
#     sequence = f"""Question : {row['prompt']}
#     A: {row['A']}
#     B: {row['B']}
#     C: {row['C']}
#     D: {row['D']}
#     E: {row['E']}"""

#     # BERT scores each label
#     res = classifier(sequence , candidate_labels)

#     # Get top 3 in here
#     top3 = res['labels'][:3]
#     t3_per_bert.append(' '.join(top3))

#     if idx % 50 == 0:
#         print(f"Done -> {idx}/500")
        
# print("Predictions Done")

In [13]:
# # Creating the BERT submission 
# submission_bert = pd.DataFrame({
#     'ID' : test_df['id'],
#     'Prediction' : t3_per_bert
# })

# # Now converting this BERT submission to a CSV file for Uploading to Kaggle as Output
# submission_bert.to_csv(
#     'submission.csv',
#     index = False
# )
# print(f"The BERT Pretrained submission is Ready\nThe Outputs look like this ->\n{submission_bert.head(10)}")

## Experiment Tracking with Weights & Biases (W&B)

To systematically track the progress of the project, all model runs, leaderboard scores, and experimental results were logged using **Weights & Biases (W&B)**.

### Purpose

As multiple approaches were explored throughout the project, it became important to maintain a centralized record of:

- Model architectures
- Kaggle leaderboard scores
- Experimental configurations
- Performance comparisons
- Training observations

W&B provided a convenient platform for organizing and visualizing these experiments.

### Logged Experiments

The following model runs were recorded:

| Run Name | Model |
|-----------|--------|
| Baseline | Frequency-Based Baseline |
| LogReg | Logistic Regression + TF-IDF |
| CatBoost | CatBoost Classifier |
| BERT-ZeroShot | Pre-Trained Transformer (Zero-Shot) |

### Benefits

Using W&B enabled:

- Reproducible experiment tracking
- Easy comparison between different approaches
- Centralized storage of leaderboard scores
- Visual analysis of model performance over time
- Better project documentation and reporting

### Outcome

All major experiments conducted during the project were successfully logged to W&B, creating a structured record of the model development journey from the initial baseline submission to advanced boosting and Transformer-based approaches.

In [14]:
# # Now we essentially 'push' the entire models summary and scores to W&B
# !pip install wandb -q
# print("W&B (also called wandb) is successfully installed")

# import wandb
# print(wandb.login())

In [15]:
# # Logging all Moodel Runs to W&B successfully
# PR_NAME = '24f3004027-t22026'

# # Note as first submission was a 0.000 formatting error that is not counted here
# # Run-1 => Baseline
# wandb.init(
#     project = PR_NAME,
#     name = "Baseline"
# )
# print("Baseline Run Initialized")
# wandb.log({
#     "kaggle_score" : 0.34871,
#     "model" : "Baseline"
# })
# print("Baseline Submission Successful")

# # Run-2 => LogReg + Tf-IDf
# wandb.init(
#     project = PR_NAME,
#     name = "LogReg"
# )
# print("Logarithmic Regression run Initialized")
# wandb.log({
#     "kaggle_score" : 0.74064,
#     "model" : "LogReg"
# })
# print("Logarithmic Regression Run Successful")

# # Run-3 => CatBoost
# wandb.init(
#     project = PR_NAME,
#     name = "CatBoost"
# )
# print("CatBoost run Initialized")
# wandb.log({
#     "kaggle_score" : 0.74979,
#     "model" : "CatBoost"
# })
# print("CatBoost Run Successful")

# # Run-4 => BERT Pretrained Model
# wandb.init(
#     project = PR_NAME,
#     name = "BERT-ZeroShot"
# )
# print("BERT-ZeroShot run Initialized")
# wandb.log({
#     "kaggle_score" : 0.41064,
#     "model" : "BERT"
# })
# print("BERT Run Successful")
# print("Successful completion of my Day-1")
# wandb.finish()

## Phase 2: Gradient Boosting Models

### Model 2: Tuned Categorical Boosting (CatBoost)

After evaluating the initial CatBoost model, a second round of experimentation was conducted to improve performance through hyperparameter tuning. The objective was to achieve better generalization and leaderboard performance by adjusting the learning rate and the number of boosting iterations.

### Methodology

#### 1. Text Processing

The same combined textual representation used in previous experiments was retained. The prompt and answer choices were merged into a single text feature and supplied directly to CatBoost.

#### 2. Native Text Handling

CatBoost's built-in text processing capabilities were leveraged, eliminating the need for external vectorization methods such as TF-IDF. The training and test datasets were converted into CatBoost `Pool` objects to enable efficient text feature processing.

#### 3. Hyperparameter Tuning

Compared to the original CatBoost configuration, the following adjustments were introduced:

| Parameter | Original Model | Tuned Model |
|------------|---------------|-------------|
| Iterations | 500 | 700 |
| Learning Rate | 0.10 | 0.08 |
| Depth | 6 | 6 |
| Loss Function | MultiClass | MultiClass |

The lower learning rate allows the model to learn more gradually, while the increased number of iterations compensates by providing additional boosting rounds.

#### 4. Model Training

The tuned CatBoost classifier was trained on the pooled training dataset using multiclass classification as the optimization objective.

Training progress was monitored through the multiclass loss reported at regular intervals throughout the boosting process.

#### 5. Prediction Generation

For each test sample:

1. Class probabilities were generated.
2. The probabilities were sorted in descending order.
3. The three most probable answer choices were selected.
4. Predictions were formatted according to the competition submission requirements.

### Rationale

Hyperparameter tuning is a critical step in improving the performance of boosting algorithms. By reducing the learning rate and increasing the number of boosting iterations, the model is allowed to capture more nuanced patterns within the data while reducing the risk of unstable learning.

### Expected Outcome

The tuned CatBoost model aims to improve upon the performance of the initial CatBoost submission and determine whether additional boosting rounds can translate into higher leaderboard scores.

### Observations

- Lower learning rates generally produce more stable optimization.
- Increasing the number of boosting iterations often improves model capacity.
- Training time increases noticeably as the number of iterations grows.
- Careful tuning is necessary to balance predictive performance against computational cost.

In [16]:
# # Here we essentially tune the CatBoost model to our liking
# from catboost import CatBoostClassifier

# # Instantiating our model
# cb_v2 = CatBoostClassifier(
#     iterations = 700,
#     learning_rate = 0.08,
#     depth = 6,
#     loss_function = "MultiClass",
#     verbose = 100
# )
# print("Tuned CatBoost is instantiated")

# # Fitting the Instance on the pooled training data 
# cb_v2.fit(train_pool)
# print("The Model has been successfully fit on training data")

# # Predict with tuned CatBoost
# probs_cb2 = cb_v2.predict_proba(test_pool)
# classes_cb2 = cb_v2.classes_

# # Get top 3 per question
# t3_pr_cb2 = []
# for prob in probs_cb2:
#     top3_idx = prob.argsort()[::-1][:3]
#     top3_lbls = ' '.join(classes_cb2[top3_idx])
#     t3_pr_cb2.append(top3_lbls)

# print("The Model is Ready !!!")

In [17]:
# submission_cb2 = pd.DataFrame({
#     'ID': test_df['id'],
#     'Prediction': t3_pr_cb2
# })
# print("Conversion to the DataFrame successful")
# submission_cb2.to_csv(
#     'submission.csv', 
#     index=False
# )
# print(f"Submission ready!\n{submission_cb2.head(10)}")

In [18]:
# Now we essentially do the Neural
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.sparse import issparse

# Convert TF-IDF sparse matrix to dense torch tensors
# Why? PyTorch doesn't understand sparse matrices, it needs regular arrays
if issparse(x_train):
    X_train_dense = torch.tensor(x_train.toarray(), dtype=torch.float32)
    X_test_dense = torch.tensor(x_test.toarray(), dtype=torch.float32)
else:
    X_train_dense = torch.tensor(x_train, dtype=torch.float32)
    X_test_dense = torch.tensor(x_test, dtype=torch.float32)

# Convert labels (A,B,C,D,E) to numbers (0,1,2,3,4)
# Why? Neural networks only understand numbers, not letters
label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
y_train_tensor = torch.tensor(
    [label_map[l] for l in y_train],
    dtype=torch.long  # long = integer type, required for classification
)

print("Data prepared!")
print(f"X_train shape: {X_train_dense.shape}")
print(f"y_train shape: {y_train_tensor.shape}")

Data prepared!
X_train shape: torch.Size([2000, 2940])
y_train shape: torch.Size([2000])


In [19]:
# Now we build the network

class MCQNet(nn.Module):
    # nn.Module is the base class for all neural networks in PyTorch
    # Every custom network must inherit from it
    
    def __init__(self):
        super().__init__()  # initializes the parent nn.Module class
        
        self.network = nn.Sequential(
            # Sequential means layers run one after another in order
            
            nn.Linear(2940, 256),  # Input layer: 2940 features → 256 neurons
            nn.ReLU(),             # Kill negative numbers
            
            nn.Linear(256, 64),    # Hidden layer: 256 → 64 neurons
            nn.ReLU(),             # Kill negative numbers again
            
            nn.Linear(64, 5)       # Output layer: 64 → 5 (one per class A,B,C,D,E)
            # No ReLU here — we want raw scores for loss calculation
        )
    
    def forward(self, x):
        # forward() defines how data flows through the network
        # x = input data
        return self.network(x)

# Instantiate the model
model_nn = MCQNet()
print(model_nn)
print("Neural Network built successfully!")

MCQNet(
  (network): Sequential(
    (0): Linear(in_features=2940, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=5, bias=True)
  )
)
Neural Network built successfully!


In [20]:
# Here we essentially train the values

# Loss function - measures how wrong the model is
# CrossEntropyLoss is standard for multi-class classification
criterion = nn.CrossEntropyLoss()

# Optimizer - updates weights to fix mistakes
# lr = learning rate, how big each fix step is
optimizer = optim.Adam(
    model_nn.parameters(),  # which weights to update
    lr=0.001                # learning rate - small steps = careful learning
)

# Number of times to go through ALL training data
epochs = 50

for epoch in range(epochs):
    
    # Forward pass - data goes through network, get predictions
    outputs = model_nn(X_train_dense)
    
    # Calculate loss - how wrong were we?
    loss = criterion(outputs, y_train_tensor)
    
    # Backward pass - figure out which weights caused the error
    optimizer.zero_grad()  # clear previous gradients first
    loss.backward()        # calculate new gradients
    optimizer.step()       # update weights
    
    # Print progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/50 → Loss: {loss.item():.4f}")

print("Training Complete!")

Epoch 10/50 → Loss: 1.5497
Epoch 20/50 → Loss: 1.3479
Epoch 30/50 → Loss: 0.9273
Epoch 40/50 → Loss: 0.4342
Epoch 50/50 → Loss: 0.1297
Training Complete!


In [21]:
# Predicting the values

# torch.no_grad() - don't calculate gradients during prediction
# Why? We're not training anymore, just predicting. Saves memory.
with torch.no_grad():
    test_outputs = model_nn(X_test_dense)
    
    # Softmax converts raw scores to probabilities
    probs_nn = torch.softmax(test_outputs, dim=1).numpy()

# Class order is always A,B,C,D,E
classes_nn = ['A', 'B', 'C', 'D', 'E']

# Get top 3 per question
t3_pr_nn = []
for prob in probs_nn:
    top3_idx = prob.argsort()[::-1][:3]
    top3_lbls = ' '.join([classes_nn[i] for i in top3_idx])
    t3_pr_nn.append(top3_lbls)
print("Predicting phase complete")

Predicting phase complete


In [22]:
# Now we create the submission
submission_nn = pd.DataFrame({
    'ID': test_df['id'],
    'Prediction': t3_pr_nn
})
submission_nn.to_csv('submission.csv', index=False)
print(f"Submission ready!\n{submission_nn.head(10)}")

Submission ready!
   ID Prediction
0   1      A D E
1   2      B D C
2   3      B D C
3   4      E C A
4   5      C E B
5   6      D A B
6   7      E C A
7   8      B D C
8   9      C B E
9  10      B D C
